# Telco Customer Churn: step-by-step walkthrough

**Goal:** predict which telecom customers are about to leave (*churn*), understand why, and turn that into retention actions.

This notebook explains every step in plain language. The reusable code lives in `src/`; here we call it piece by piece so you can see what happens at each stage.

> Before running: download the Kaggle dataset into `data/raw/` (see the README).

## Step 0: Business problem
Getting a new customer costs much more than keeping one. If we can **rank customers by their chance of leaving**, the retention team can call the riskiest ones first instead of calling everyone.

*Success looks like:* catching most real churners (high **recall**) without flooding the team with false alarms (decent **precision**).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # lets the notebook import the `src` package

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_cleaning import load_raw, clean_data
from src.features import add_features, split_xy
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## Step 1: Load the data
Each row is one customer: who they are, which services they have, how they pay, and whether they left (`Churn`).

In [ ]:
raw = load_raw()
print(raw.shape)
raw.head()

## Step 2: Inspect the data
We look for problems *before* modelling: wrong data types, missing values, odd categories.

In [ ]:
raw.info()

In [ ]:
# TotalCharges should be a number but is stored as text. Why? Some cells are blank strings.
blank = raw["TotalCharges"].astype(str).str.strip().eq("")
print("Blank TotalCharges:", blank.sum())
raw.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]].head()

These blank rows all have `tenure = 0`: brand-new customers who have not been billed yet. Their total so far is really **0**, so we fill it with 0.

## Step 3: Clean the data
`clean_data()` does four things: removes duplicates, converts `TotalCharges` to numbers, merges *"No internet service"* / *"No phone service"* into plain *"No"*, and turns `Churn` into 1 (left) / 0 (stayed).

In [ ]:
df = clean_data(raw)
print("Missing values left:", int(df.isna().sum().sum()))
print(f"Overall churn rate: {df['Churn'].mean():.1%}")
df.head()

## Step 4: Exploratory Data Analysis (EDA)
EDA = asking the data questions with charts. Note the **class imbalance**: far fewer customers churn than stay. That is why accuracy alone is misleading.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x="Churn", ax=axes[0]); axes[0].set_title("Churn (0 = stayed, 1 = left)")
rate = df.groupby("Contract")["Churn"].mean().mul(100)
sns.barplot(x=rate.index, y=rate.values, ax=axes[1]); axes[1].set_title("Churn rate (%) by contract")
sns.histplot(data=df, x="tenure", hue="Churn", element="step", ax=axes[2]); axes[2].set_title("Tenure by churn")
plt.tight_layout(); plt.show()

In [ ]:
for col in ["InternetService", "PaymentMethod", "TechSupport"]:
    print(df.groupby(col)["Churn"].agg(churn_rate="mean", customers="size").round(3).sort_values("churn_rate", ascending=False), "\n")

**What to look for:** month-to-month contracts, short tenure, fiber internet, electronic-check payments and no tech support usually go with higher churn. Always confirm with *your* run.

## Step 5: Feature engineering
New columns that express business ideas the model can use directly:
- `tenure_group`: new vs loyal customers
- `num_addon_services`: how many extras they have (more = stickier)
- `has_autopay`, `has_family`: lower-friction, more-stable customers
- `monthly_per_service`: price paid per service

In [ ]:
df = add_features(df)
df[["tenure", "tenure_group", "num_addon_services", "monthly_per_service", "has_autopay", "has_family"]].head()

## Step 6: Train/test split
We hide 20% of customers (the **test set**) and only use them at the very end. This mimics predicting on future customers. `stratify=y` keeps the churn share equal in both parts.

In [ ]:
from sklearn.model_selection import train_test_split
X, y = split_xy(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, f"| churn share train {y_train.mean():.1%} / test {y_test.mean():.1%}")

## Step 7: Preprocess and train three models
Models only understand numbers, so the **preprocessor** scales numeric columns and one-hot encodes text categories. We wrap it and the model in one **Pipeline**, so exactly the same steps run in training and in the app.

We compare **Logistic Regression** (simple, explainable), **Random Forest** (many decision trees voting) and **XGBoost** (trees that learn from each other's mistakes).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from src.train import build_preprocessor, get_models

spw = (y_train == 0).sum() / (y_train == 1).sum()   # used by XGBoost to weight churners more
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipes = {}
for name, model in get_models(spw).items():
    pipe = Pipeline([("preprocess", build_preprocessor()), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    pipe.fit(X_train, y_train)
    pipes[name] = pipe
    print(f"{name:20s} cross-validated ROC-AUC: {scores.mean():.3f} +/- {scores.std():.3f}")

## Step 8: Evaluate on the hidden test set
- **Accuracy**: share of correct predictions
- **Precision**: when we flag churn, how often is it right?
- **Recall**: of the real churners, how many did we catch?
- **F1**: balance of precision and recall
- **ROC-AUC**: how well we rank risky customers above safe ones

In [ ]:
from src.train import evaluate
results = {name: evaluate(pipe, X_test, y_test) for name, pipe in pipes.items()}
pd.DataFrame({n: r["metrics"] for n, r in results.items()}).T.round(3)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pipe) in zip(axes, pipes.items()):
    ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test, display_labels=["Stayed", "Churned"], cmap="Purples", colorbar=False, ax=ax)
    ax.set_title(name)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
for name, r in results.items():
    plt.plot(r["roc_curve"]["fpr"], r["roc_curve"]["tpr"], label=f"{name} (AUC {r['metrics']['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey", label="Random guessing")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.legend(); plt.title("ROC curves"); plt.show()

## Step 9: Feature importance: what drives churn?
Tree models report how much each feature helped; Logistic Regression reports coefficients (positive = raises churn risk). Remember: this shows **association, not proof of cause**.

In [ ]:
from src.train import feature_importance
best = max(results, key=lambda n: results[n]["metrics"]["roc_auc"])
imp = pd.DataFrame(feature_importance(pipes[best])).head(15)
plt.figure(figsize=(7, 5))
sns.barplot(data=imp, x="importance", y="feature", color="#6366F1")
plt.title(f"Top 15 drivers: {best}"); plt.show()

## Step 10: Save and ship
`python -m src.train` repeats all of this, picks the winner using cross-validation, and saves `models/churn_pipeline.joblib` plus `models/metrics.json`. Then launch the app:

```bash
streamlit run app/main.py
```